# Track 1 — Analytics Layer
UPI Fraud Ring & Merchant Analytics — TransOrg AgentIQ Datathon

This notebook turns the four cleaned files into a queryable **star-schema SQLite database**
(`track1_analytics.db`) plus two derived metrics tables, and documents the exact formula
behind every metric — especially **Fraud Risk Score** and **Churn**.

**Inputs:** `track1_upi_transactions_clean.csv`, `track1_kyc_records_cleaned.csv`,
`track1_merchants_master_cleaned.csv`, `json_cleaned.csv` (chargebacks).

**Sections**
1. Load & profile
2. Entity resolution (a data-quality issue the "cleaned" files still have)
3. Star schema: dimension & fact tables
4. Metric definitions (exact formulas) + computation
5. Load into SQLite + example queries


In [1]:
import pandas as pd
import numpy as np
import sqlite3, os, glob


DATA_DIR = "."
DB_PATH = "track1_analytics.db"

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

def find_file(keywords):
    """Find a CSV in DATA_DIR whose filename contains all given keywords
    (case-insensitive). Raises a clear error listing what IS there if nothing matches,
    instead of a raw FileNotFoundError."""
    candidates = glob.glob(os.path.join(DATA_DIR, "*.csv"))
    for path in candidates:
        name = os.path.basename(path).lower()
        if all(k.lower() in name for k in keywords):
            return path
    available = "\n  ".join(os.path.basename(c) for c in candidates) or "(no .csv files found in DATA_DIR at all)"
    raise FileNotFoundError(
        f"Could not find a CSV matching keywords {keywords} in '{os.path.abspath(DATA_DIR)}'.\n"
        f"CSV files actually present there:\n  {available}\n"
        f"Fix: either rename/move the right file into DATA_DIR, or edit DATA_DIR above to point "
        f"at the correct folder."
    )

txn_path = find_file(["upi", "transaction"])
kyc_path = find_file(["kyc"])
mer_path = find_file(["merchant"])
cbk_path = find_file(["json", "clean"])

print("Using files:")
print(" transactions:", txn_path)
print(" kyc         :", kyc_path)
print(" merchants   :", mer_path)
print(" chargebacks :", cbk_path)
print()

txn = pd.read_csv(txn_path, parse_dates=["timestamp"])
kyc = pd.read_csv(kyc_path, parse_dates=["signup_timestamp", "date_of_birth"])
mer = pd.read_csv(mer_path, parse_dates=["onboarding_date"])
cbk = pd.read_csv(cbk_path,
                   parse_dates=["transaction_timestamp", "reported_timestamp", "bank_response_timestamp"])

print("transactions:", txn.shape)
print("kyc_records :", kyc.shape)
print("merchants   :", mer.shape)
print("chargebacks :", cbk.shape)

Using files:
 transactions: .\track1_upi_transactions_clean.csv
 kyc         : .\track1_kyc_records_cleaned.csv
 merchants   : .\track1_merchants_master_cleaned.csv
 chargebacks : .\json_cleaned.csv

transactions: (20000, 13)
kyc_records : (35878, 19)
merchants   : (6000, 18)
chargebacks : (2800, 19)


## 2. Entity resolution — a data-quality issue that survived cleaning

Before building the schema I checked whether `user_id` and `merchant_id` are safe
**primary keys**. They are not, and this matters for every metric downstream:

- **KYC `user_id`** appears in four different string formats
  (`USR#####`, `USR-#####`, `USR_#####`, bare `#####`). I tested the obvious fix —
  stripping the separators and zero-padding to one canonical form — and it's **wrong**:
  doing that merges rows with different `full_name` and different `pan` values ~85% of
  the time. These aren't repeat KYC attempts by the same person; they're **different
  people whose IDs collide**. Even the exact same raw string (`user_id` with no
  reformatting at all) is shared by unrelated people in ~85% of its duplicate groups.
- **Merchant master `merchant_id`** has an explicit `id_collision_flag` column for
  exactly this reason: `MCH1007_1` and `MCH1007_2` are two different merchants that
  both reduce to base id `MCH1007` — the id transactions actually reference.

**Decision:** I don't silently merge colliding identities (that would fabricate a
single fake profile out of two real ones). Instead I group by the id, and:
- if every row in the group agrees on `full_name`/`pan` (or there's only one row) →
  treat it as one clean identity;
- if they disagree → keep the group as **one row per id** (so transactions still
  join), take the *worst-case* KYC status / risk segment across the colliding
  identities (risk-conservative for a fraud use case), and set
  `identity_ambiguous = True` so any score built on it is auditable.

This means `dim_users` / `dim_merchants` have exactly one row per id used in the
transaction data, and every fraud/churn score inherited from an ambiguous id carries
a visible flag rather than a false precision.

In [2]:
risk_rank = {"LOW": 0, "MEDIUM": 1, "UNKNOWN": 1, "HIGH": 2}
status_rank = {"Verified": 0, "Pending": 1, "Failed": 2, "Rejected": 3}  # higher = worse

kyc["_risk_rank"] = kyc["risk_segment"].map(risk_rank).fillna(1)
kyc["_status_rank"] = kyc["kyc_status"].map(status_rank).fillna(1)

def resolve_user_group(g):
    n_identities = g["pan"].dropna().nunique() or g["full_name"].nunique()
    ambiguous = len(g) > 1 and (g["full_name"].nunique() > 1 or g["pan"].dropna().nunique() > 1)
    worst_status = g.loc[g["_status_rank"].idxmax(), "kyc_status"]
    worst_risk = g.loc[g["_risk_rank"].idxmax(), "risk_segment"]
    latest = g.sort_values("signup_timestamp", na_position="first").iloc[-1]
    return pd.Series({
        "full_name": latest["full_name"] if not ambiguous else "AMBIGUOUS_ID",
        "city": latest["city"], "state": latest["state"],
        "monthly_income": g["monthly_income"].median(),
        "occupation": latest["occupation"] if not ambiguous else "AMBIGUOUS_ID",
        "signup_timestamp": g["signup_timestamp"].min(),
        "kyc_status": worst_status, "risk_segment": worst_risk,
        "conflict_count": g["conflict_count"].max(),
        "has_conflict": int(g["has_conflict"].max() or ambiguous),
        "age": g["age"].median(), "account_age_days": g["account_age_days"].max(),
        "n_kyc_rows": len(g), "n_distinct_identities": n_identities,
        "identity_ambiguous": ambiguous,
    })

dim_users = kyc.groupby("user_id", as_index=True).apply(resolve_user_group).reset_index()
print(f"dim_users: {dim_users.shape[0]} unique ids | {dim_users['identity_ambiguous'].sum()} flagged ambiguous "
      f"({dim_users['identity_ambiguous'].mean():.1%})")
dim_users.head(3)

dim_users: 30598 unique ids | 3954 flagged ambiguous (12.9%)


,user_id,full_name,city,state,monthly_income,occupation,signup_timestamp,kyc_status,risk_segment,conflict_count,has_conflict,age,account_age_days,n_kyc_rows,n_distinct_identities,identity_ambiguous
0,10098,xiti ben,Chennai,Tamil Nadu,NaN,Gig Worker,NaT,Rejected,LOW,1,1,64.922656,NaN,1,1,False
1,10100,Gavin Mani,Jalandhar,Punjab,79104.0,Student,NaT,Verified,LOW,0,0,23.709788,NaN,1,1,False
2,10107,KALPIT NARAYAN,Hyderabad,Telangana,47795.0,Self Employed,NaT,Verified,LOW,0,0,44.476386,NaN,1,1,False


In [3]:
mer["base_merchant_id"] = mer["merchant_id"].str.split("_").str[0]
status_order = {"Active": 0, "Inactive": 1, "Suspended": 2, "Closed": 3}

def resolve_merchant_group(g):
    ambiguous = len(g) > 1
    any_suspended = bool(g["is_suspended_or_blocked"].any())
    g = g.copy()
    g["_st_rank"] = g["merchant_status"].map(status_order).fillna(0)
    worst = g.loc[g["_st_rank"].idxmax()]
    return pd.Series({
        "merchant_name": worst["merchant_name"] if not ambiguous else "AMBIGUOUS_ID",
        "mcc": worst["mcc"],
        "merchant_category": worst["merchant_category"] if not ambiguous else "AMBIGUOUS_ID",
        "business_type": worst["business_type"], "city": worst["city"], "state": worst["state"],
        "onboarding_date": g["onboarding_date"].min(),
        "merchant_status": worst["merchant_status"],
        "is_suspended_or_blocked": any_suspended,
        "declared_avg_ticket_size": g["declared_avg_ticket_size"].median(),
        "category_median_ticket_size": g["category_median_ticket_size"].median(),
        "n_master_rows": len(g), "identity_ambiguous": ambiguous,
    })

dim_merchants = mer.groupby("base_merchant_id", as_index=True).apply(resolve_merchant_group).reset_index()
dim_merchants = dim_merchants.rename(columns={"base_merchant_id": "merchant_id"})
print(f"dim_merchants: {dim_merchants.shape[0]} unique ids | {dim_merchants['identity_ambiguous'].sum()} flagged ambiguous "
      f"({dim_merchants['identity_ambiguous'].mean():.1%})")
dim_merchants.head(3)

dim_merchants: 4343 unique ids | 1310 flagged ambiguous (30.2%)


,merchant_id,merchant_name,mcc,merchant_category,business_type,city,state,onboarding_date,merchant_status,is_suspended_or_blocked,declared_avg_ticket_size,category_median_ticket_size,n_master_rows,identity_ambiguous
0,MCH1001,Anand-Kothari,NaN,Grocery,Partnership,Pune,Maharashtra,2026-02-17 06:54:00,Active,False,332.55,1233.545,1,False
1,MCH1002,"Apte, Saha and Edwin",5812.0,Restaurant,Private Limited,Bengaluru,Karnataka,2026-07-01 09:01:00,Active,False,1671.20,1327.310,1,False
2,MCH1003,"Nayar, Batta and Barad",5699.0,Apparel,Private Limited,Chennai,Tamil Nadu,NaT,Active,False,792.91,1252.430,1,False


## 3. Star schema

| Table | Grain | Role |
|---|---|---|
| `dim_users` | 1 row / resolved `user_id` | KYC attributes, risk segment, ambiguity flag |
| `dim_merchants` | 1 row / resolved `merchant_id` | category, status, ambiguity flag |
| `fact_transactions` | 1 row / `txn_id` | the UPI transaction ledger |
| `fact_chargebacks` | 1 row / `complaint_id` | disputes, linked to `txn_id` / `user_id` / `merchant_id` |
| `merchant_metrics` | 1 row / `merchant_id` | derived: rates, fraud risk score, churn flag |
| `user_metrics` | 1 row / `user_id` | derived: rates, fraud risk score |

`fact_transactions` and `fact_chargebacks` already had single, consistently formatted
ids in the cleaned files, so they're used as-is — only the two dimension tables needed
entity resolution.

In [4]:
fact_txn = txn[["txn_id", "timestamp", "user_id", "merchant_id", "amount", "utr",
                "mcc", "status", "is_reversal", "utr_missing", "mcc_missing"]].copy()

fact_cbk = cbk[["complaint_id", "txn_id", "user_id", "merchant_id", "transaction_timestamp",
                "reported_timestamp", "reporting_delay_days", "disputed_amount",
                "reason_code_bucket", "resolution_status", "severity", "channel"]].copy()

mcov = fact_txn["merchant_id"].isin(dim_merchants["merchant_id"]).mean()
ucov = fact_txn["user_id"].isin(dim_users["user_id"]).mean()
print(f"fact_transactions: {fact_txn.shape[0]:,} rows")
print(f"fact_chargebacks : {fact_cbk.shape[0]:,} rows")
print(f"Txn rows with a matching merchant-master record: {mcov:.1%}")
print(f"Txn rows with a matching KYC record:              {ucov:.1%}")
print("(No master/KYC row just means we fall back to transaction-only rates for that id —")
print(" it does not break the join, it limits which metrics can be computed for it.)")

fact_transactions: 20,000 rows
fact_chargebacks : 2,800 rows
Txn rows with a matching merchant-master record: 48.2%
Txn rows with a matching KYC record:              28.0%
(No master/KYC row just means we fall back to transaction-only rates for that id —
 it does not break the join, it limits which metrics can be computed for it.)


## 4. Metric definitions

### 4.1 Merchant Fraud Risk Score (0–100)

For merchant *m*, over the full observation window:

- `N_m` = total transactions, `Success_m` = count(status = SUCCESS)
- `failure_rate_m = Failed_m / N_m`
- `reversal_rate_m = Reversals_m / N_m`
- `chargeback_rate_m = Chargebacks_m / max(Success_m, 1)` — disputes per completed
  transaction (denominator is completed transactions, since only those can be disputed)
- `severity_weighted_rate_m = Σ w(severity_i) / max(Success_m, 1)`, with
  `w = {Low:1, Medium:2, High:3, Critical:5}` — a merchant with a few *Critical*
  disputes should rank above one with many *Low* ones
- `high_risk_user_share_m` = share of *m*'s distinct counterpart users whose KYC
  `risk_segment = HIGH`

Each rate is min-max normalized across merchants with ≥ 5 transactions
(`x_norm = (x - min) / (max - min)`, small samples excluded so one unlucky
transaction can't fake a 100% chargeback rate) and combined:

```
FraudRiskScore_m = 100 × ( 0.30·chargeback_norm + 0.20·severity_norm
                          + 0.20·failure_norm   + 0.15·reversal_norm
                          + 0.15·high_risk_user_share_norm )
```

with a **hard floor of 90** if the merchant master already marks the merchant
`Suspended` / `is_suspended_or_blocked` — a confirmed suspension should never be
out-ranked by the formula.

### 4.2 User Fraud Risk Score (0–100)

Analogous, at user grain:

```
UserFraudRiskScore_u = 100 × ( 0.30·chargeback_rate_norm + 0.15·failure_rate_norm
                              + 0.15·reversal_rate_norm  + 0.25·kyc_risk_score
                              + 0.15·conflict_score )
```
where `kyc_risk_score = {LOW:0, MEDIUM:0.5, UNKNOWN:0.5, HIGH:1}` and
`conflict_score = min(conflict_count / 3, 1)` (KYC's own `conflict_count` field,
capped at 3+). Floor of 80 if `kyc_status ∈ {Rejected, Failed}`.

*Caveat stated plainly:* a chargeback tells you a dispute happened on the user's
transaction, not who was at fault — it's a reasonable risk signal (repeat disputes
correlate with both fraud and being targeted by it) but not proof of wrongdoing.

### 4.3 Merchant Churn

Snapshot date `T_max` = latest transaction timestamp in the data. Churn window
`W = 30 days` (the data spans ~90 days, so this gives one clean baseline period
and one clean recent period):

```
active_in_baseline_m = 1 if m had ≥1 txn in [T_max − 60d, T_max − 30d)
is_churned_m          = 1 if active_in_baseline_m = 1  AND  m had 0 txns in [T_max − 30d, T_max]

Merchant Churn Rate = Σ is_churned_m / Σ active_in_baseline_m
```

i.e. of the merchants who were transacting a month ago, what share have gone
completely silent in the most recent 30 days. `W` is a parameter, not a constant —
30 days is the natural choice given the ~90-day window; it's the one line to change
if the dashboard needs a different definition.

In [5]:
SEVERITY_WEIGHT = {"Low": 1, "Medium": 2, "High": 3, "Critical": 5}
RISK_WEIGHT = {"LOW": 0.0, "MEDIUM": 0.5, "HIGH": 1.0, "UNKNOWN": 0.5}
CHURN_WINDOW_DAYS = 30
ELIGIBLE_N = 5
T_MAX = fact_txn["timestamp"].max()

def minmax(s):
    lo, hi = s.min(), s.max()
    return pd.Series(0.0, index=s.index) if hi - lo == 0 else (s - lo) / (hi - lo)

# ---- merchant aggregates ----
g = fact_txn.groupby("merchant_id")
mm = g.agg(txn_count=("txn_id", "count"),
           success_count=("status", lambda s: (s == "SUCCESS").sum()),
           failed_count=("status", lambda s: (s == "FAILED").sum()),
           reversal_count=("is_reversal", "sum"),
           last_txn_date=("timestamp", "max"),
           distinct_users=("user_id", "nunique")).reset_index()

cbk_by_mer = fact_cbk.groupby("merchant_id").agg(
    chargeback_count=("complaint_id", "count"),
    severity_score=("severity", lambda s: s.map(SEVERITY_WEIGHT).sum())).reset_index()
mm = mm.merge(cbk_by_mer, on="merchant_id", how="left")
mm[["chargeback_count", "severity_score"]] = mm[["chargeback_count", "severity_score"]].fillna(0)

ut = fact_txn.merge(dim_users[["user_id", "risk_segment"]], on="user_id", how="left")
hr = ut.groupby("merchant_id").apply(
    lambda d: (d.drop_duplicates("user_id")["risk_segment"] == "HIGH").mean()
).rename("high_risk_user_share").reset_index()
mm = mm.merge(hr, on="merchant_id", how="left")
mm["high_risk_user_share"] = mm["high_risk_user_share"].fillna(0)

mm["success_denom"] = mm["success_count"].clip(lower=1)
mm["failure_rate"] = mm["failed_count"] / mm["txn_count"]
mm["reversal_rate"] = mm["reversal_count"] / mm["txn_count"]
mm["chargeback_rate"] = mm["chargeback_count"] / mm["success_denom"]
mm["severity_weighted_rate"] = mm["severity_score"] / mm["success_denom"]

mm = mm.merge(dim_merchants[["merchant_id", "is_suspended_or_blocked", "identity_ambiguous"]],
              on="merchant_id", how="left")
mm["is_suspended_or_blocked"] = mm["is_suspended_or_blocked"].fillna(False)

elig = mm["txn_count"] >= ELIGIBLE_N
for src, dst in [("chargeback_rate","cb_norm"), ("severity_weighted_rate","sev_norm"),
                  ("failure_rate","fail_norm"), ("reversal_rate","rev_norm"),
                  ("high_risk_user_share","hr_norm")]:
    mm[dst] = 0.0
    mm.loc[elig, dst] = minmax(mm.loc[elig, src])

mm["fraud_risk_score"] = 100 * (0.30*mm["cb_norm"] + 0.20*mm["sev_norm"] + 0.20*mm["fail_norm"]
                                 + 0.15*mm["rev_norm"] + 0.15*mm["hr_norm"])
mm.loc[mm["is_suspended_or_blocked"], "fraud_risk_score"] = \
    mm.loc[mm["is_suspended_or_blocked"], "fraud_risk_score"].clip(lower=90)

# ---- churn ----
recent_start = T_MAX - pd.Timedelta(days=CHURN_WINDOW_DAYS)
baseline_start = T_MAX - pd.Timedelta(days=2*CHURN_WINDOW_DAYS)
active_baseline = set(fact_txn.loc[(fact_txn["timestamp"] >= baseline_start) &
                                    (fact_txn["timestamp"] < recent_start), "merchant_id"])
active_recent = set(fact_txn.loc[fact_txn["timestamp"] >= recent_start, "merchant_id"])

mm["recency_days"] = (T_MAX - mm["last_txn_date"]).dt.days
mm["active_in_baseline"] = mm["merchant_id"].isin(active_baseline)
mm["is_churned"] = mm["active_in_baseline"] & ~mm["merchant_id"].isin(active_recent)

churn_rate = mm["is_churned"].sum() / max(mm["active_in_baseline"].sum(), 1)
print(f"Snapshot date T_max = {T_MAX.date()}  |  churn window = {CHURN_WINDOW_DAYS}d")
print(f"Merchant churn rate: {churn_rate:.2%}  ({mm['is_churned'].sum()} of {mm['active_in_baseline'].sum()} baseline-active merchants)")

merchant_metrics = mm[["merchant_id","txn_count","success_count","failed_count","reversal_count",
                        "chargeback_count","distinct_users","failure_rate","reversal_rate",
                        "chargeback_rate","severity_weighted_rate","high_risk_user_share",
                        "fraud_risk_score","last_txn_date","recency_days","active_in_baseline",
                        "is_churned","is_suspended_or_blocked","identity_ambiguous"]].copy()

merchant_metrics.sort_values("fraud_risk_score", ascending=False).head(10)

Snapshot date T_max = 2026-03-31  |  churn window = 30d
Merchant churn rate: 47.41%  (2246 of 4737 baseline-active merchants)


,merchant_id,txn_count,success_count,failed_count,reversal_count,chargeback_count,distinct_users,failure_rate,reversal_rate,chargeback_rate,severity_weighted_rate,high_risk_user_share,fraud_risk_score,last_txn_date,recency_days,active_in_baseline,is_churned,is_suspended_or_blocked,identity_ambiguous
2016,MCH3237,2,2,0,0,0.0,2,0.0,0.00,0.000000,0.000000,0.000000,90.0,2026-03-14 13:18:15,17,False,False,True,False
590,MCH1647,4,4,0,1,0.0,4,0.0,0.25,0.000000,0.000000,0.000000,90.0,2026-03-26 04:57:39,5,True,False,True,False
2108,MCH3351,2,0,2,0,0.0,2,1.0,0.00,0.000000,0.000000,0.000000,90.0,2026-02-04 11:55:33,55,True,True,True,False
1972,MCH3189,2,2,0,0,0.0,2,0.0,0.00,0.000000,0.000000,0.000000,90.0,2026-02-08 21:58:12,51,True,True,True,True
341,MCH1375,3,3,0,0,0.0,3,0.0,0.00,0.000000,0.000000,0.333333,90.0,2026-03-02 00:00:00,29,True,False,True,True
3833,MCH5293,2,2,0,0,0.0,2,0.0,0.00,0.000000,0.000000,0.500000,90.0,2026-03-30 19:30:05,1,False,False,True,True
3794,MCH5250,2,2,0,0,0.0,2,0.0,0.00,0.000000,0.000000,0.000000,90.0,2026-01-10 08:23:31,80,False,False,True,False
2293,MCH3558,1,1,0,0,1.0,1,0.0,0.00,1.000000,2.000000,0.000000,90.0,2026-03-07 04:58:18,24,False,False,True,True
925,MCH2020,3,3,0,0,1.0,3,0.0,0.00,0.333333,0.666667,0.333333,90.0,2026-03-12 09:53:07,19,True,False,True,True
346,MCH1380,3,3,0,0,0.0,3,0.0,0.00,0.000000,0.000000,0.000000,90.0,2026-03-17 07:51:39,14,False,False,True,True


In [6]:
# ---- user aggregates ----
gu = fact_txn.groupby("user_id")
um = gu.agg(txn_count=("txn_id","count"),
            failed_count=("status", lambda s: (s=="FAILED").sum()),
            reversal_count=("is_reversal","sum"),
            last_txn_date=("timestamp","max")).reset_index()

cbk_by_user = fact_cbk.groupby("user_id").agg(chargeback_count=("complaint_id","count")).reset_index()
um = um.merge(cbk_by_user, on="user_id", how="left")
um["chargeback_count"] = um["chargeback_count"].fillna(0)

um = um.merge(dim_users[["user_id","risk_segment","kyc_status","conflict_count","identity_ambiguous"]],
              on="user_id", how="left")

um["failure_rate"] = um["failed_count"] / um["txn_count"]
um["reversal_rate"] = um["reversal_count"] / um["txn_count"]
um["chargeback_rate"] = um["chargeback_count"] / um["txn_count"]
um["kyc_risk_score"] = um["risk_segment"].map(RISK_WEIGHT).fillna(0.5)
um["conflict_score"] = (um["conflict_count"].fillna(0) / 3).clip(upper=1)

eligu = um["txn_count"] >= ELIGIBLE_N
for src, dst in [("chargeback_rate","cb_norm"), ("failure_rate","fail_norm"), ("reversal_rate","rev_norm")]:
    um[dst] = 0.0
    um.loc[eligu, dst] = minmax(um.loc[eligu, src])

um["user_fraud_risk_score"] = 100 * (0.30*um["cb_norm"] + 0.15*um["fail_norm"] + 0.15*um["rev_norm"]
                                      + 0.25*um["kyc_risk_score"] + 0.15*um["conflict_score"])
bad_status = um["kyc_status"].isin(["Rejected","Failed"])
um.loc[bad_status, "user_fraud_risk_score"] = um.loc[bad_status, "user_fraud_risk_score"].clip(lower=80)

user_metrics = um[["user_id","txn_count","failed_count","reversal_count","chargeback_count",
                    "failure_rate","reversal_rate","chargeback_rate","risk_segment","kyc_status",
                    "conflict_count","user_fraud_risk_score","last_txn_date","identity_ambiguous"]].copy()

user_metrics.sort_values("user_fraud_risk_score", ascending=False).head(10)

,user_id,txn_count,failed_count,reversal_count,chargeback_count,failure_rate,reversal_rate,chargeback_rate,risk_segment,kyc_status,conflict_count,user_fraud_risk_score,last_txn_date,identity_ambiguous
1694,USR18341,1,0,0,0.0,0.0,0.0,0.0,HIGH,Rejected,1.0,80.0,2026-01-27 12:43:32,False
17866,USR99944,1,0,0,0.0,0.0,0.0,0.0,MEDIUM,Rejected,10.0,80.0,2026-03-12 14:52:31,True
1882,USR19269,2,0,0,0.0,0.0,0.0,0.0,MEDIUM,Rejected,10.0,80.0,2026-02-19 10:25:31,True
1908,USR19402,1,0,0,0.0,0.0,0.0,0.0,LOW,Failed,0.0,80.0,2026-01-31 12:50:12,False
17588,USR98500,1,0,0,0.0,0.0,0.0,0.0,MEDIUM,Rejected,6.0,80.0,2026-02-09 22:13:59,True
1957,USR19633,1,1,0,0.0,1.0,0.0,0.0,LOW,Rejected,0.0,80.0,2026-01-17 00:28:16,False
1984,USR19784,1,1,0,0.0,1.0,0.0,0.0,LOW,Failed,0.0,80.0,2026-01-29 23:44:13,False
1975,USR19749,1,1,0,0.0,1.0,0.0,0.0,MEDIUM,Failed,0.0,80.0,2026-02-20 18:52:50,False
2000,USR19877,1,0,0,0.0,0.0,0.0,0.0,LOW,Failed,0.0,80.0,2026-02-18 22:39:42,False
2031,USR20036,1,0,0,0.0,0.0,0.0,0.0,HIGH,Rejected,0.0,80.0,2026-03-20 00:37:10,False


## 5. Persist to SQLite + example queries

Everything below lands in `track1_analytics.db` — six indexed tables, ready to query
directly with SQL (for the dashboard layer) or reopen with `pd.read_sql`.

In [7]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
conn = sqlite3.connect(DB_PATH)

dim_users.to_sql("dim_users", conn, index=False)
dim_merchants.to_sql("dim_merchants", conn, index=False)
fact_txn.to_sql("fact_transactions", conn, index=False)
fact_cbk.to_sql("fact_chargebacks", conn, index=False)
merchant_metrics.to_sql("merchant_metrics", conn, index=False)
user_metrics.to_sql("user_metrics", conn, index=False)

conn.executescript('''
CREATE INDEX idx_txn_user ON fact_transactions(user_id);
CREATE INDEX idx_txn_merchant ON fact_transactions(merchant_id);
CREATE INDEX idx_txn_ts ON fact_transactions(timestamp);
CREATE INDEX idx_cbk_txn ON fact_chargebacks(txn_id);
CREATE INDEX idx_cbk_merchant ON fact_chargebacks(merchant_id);
CREATE INDEX idx_cbk_user ON fact_chargebacks(user_id);
CREATE INDEX idx_mm_merchant ON merchant_metrics(merchant_id);
CREATE INDEX idx_um_user ON user_metrics(user_id);
''')
conn.commit()

print("Tables in", DB_PATH, ":")
print(pd.read_sql("SELECT name, type FROM sqlite_master WHERE type='table'", conn))

Tables in track1_analytics.db :
                name   type
0          dim_users  table
1      dim_merchants  table
2  fact_transactions  table
3   fact_chargebacks  table
4   merchant_metrics  table
5       user_metrics  table


In [8]:
# Example: top 10 highest-risk merchants with category context, min 5 txns
q = '''
SELECT mm.merchant_id, dm.merchant_name, dm.merchant_category, dm.merchant_status,
       mm.txn_count, ROUND(mm.chargeback_rate,3) AS chargeback_rate,
       ROUND(mm.failure_rate,3) AS failure_rate, ROUND(mm.fraud_risk_score,1) AS fraud_risk_score,
       mm.identity_ambiguous
FROM merchant_metrics mm
JOIN dim_merchants dm ON dm.merchant_id = mm.merchant_id
WHERE mm.txn_count >= 5
ORDER BY mm.fraud_risk_score DESC
LIMIT 10;
'''
pd.read_sql(q, conn)

,merchant_id,merchant_name,merchant_category,merchant_status,txn_count,chargeback_rate,failure_rate,fraud_risk_score,identity_ambiguous
0,MCH1073,"Dasgupta,RayandDara",Pharmacy,Suspended,6,0.0,0.167,90.0,0
1,MCH1349,AMBIGUOUS_ID,AMBIGUOUS_ID,Suspended,5,0.0,0.000,90.0,1
2,MCH1848,AMBIGUOUS_ID,AMBIGUOUS_ID,Suspended,5,0.0,0.200,90.0,1
3,MCH2083,Patla-Basak,Pharmacy,Suspended,5,0.0,0.000,90.0,0
4,MCH2516,Ben-Varghese,Grocery,Suspended,5,0.0,0.000,90.0,0
5,MCH2571,Deep-Chakraborty,Restaurant,Suspended,5,0.0,0.000,90.0,0
6,MCH3559,Chokshi Inc,Restaurant,Suspended,6,0.0,0.167,90.0,0
7,MCH3715,BANSAL INC,Books & Stationery,Suspended,5,0.0,0.200,90.0,0
8,MCH3722,AMBIGUOUS_ID,AMBIGUOUS_ID,Suspended,5,0.0,0.000,90.0,1
9,MCH4909,"Sharaf, Lalla and Rattan",Books & Stationery,Suspended,5,0.5,0.000,90.0,0


In [9]:
# Example: churned merchants by category
q = '''
SELECT dm.merchant_category, COUNT(*) AS churned_merchants
FROM merchant_metrics mm
JOIN dim_merchants dm ON dm.merchant_id = mm.merchant_id
WHERE mm.is_churned = 1
GROUP BY dm.merchant_category
ORDER BY churned_merchants DESC;
'''
pd.read_sql(q, conn)

,merchant_category,churned_merchants
0,AMBIGUOUS_ID,310
1,Hotel,93
2,Misc Retail,82
3,Books & Stationery,80
4,Pharmacy,76
5,Grocery,70
6,Apparel,69
7,Transportation,68
8,Restaurant,68
9,Department Store,51


In [10]:
conn.close()
print("Analytics layer complete: dim_users, dim_merchants, fact_transactions,")
print("fact_chargebacks, merchant_metrics, user_metrics — all indexed and queryable.")

Analytics layer complete: dim_users, dim_merchants, fact_transactions,
fact_chargebacks, merchant_metrics, user_metrics — all indexed and queryable.
